# Erase Dutch from a language model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThatE10/rspd/blob/main/examples/erase_dutch.ipynb)

This notebook removes one language, Dutch, from an open 4B language model and leaves English, German and French in place. There is no training: SRSPD solves for the part of each layer that Dutch uses and the other languages do not, and subtracts it.

**To run it:** on Colab, choose *Runtime → Change runtime type → T4 GPU* (the free one), then *Runtime → Run all*. It takes about 15 minutes, ten of them the solve. You do not need an account anywhere: the model and the datasets are open.

What happens:

1. Load the model and measure how well it reads four languages.
2. Collect 1,000 tokens of Dutch (the **target**) and 20,000 tokens of English, German and chat (the **retained** set).
3. Fit: one call.
4. Subtract the components and measure again.
5. Turn the dose up and down without solving again, then put the model back exactly as it was.

In [ ]:
# Installs what is missing (on Colab). Nothing happens if it is already installed.
import importlib.util, subprocess, sys

need = [("rspd", "rspd[hf] @ git+https://github.com/ThatE10/rspd"), ("datasets", "datasets")]
missing = [spec for name, spec in need if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

## Settings

The sizes below are what fits a free Colab GPU (16 GB) in about ten minutes of solving. `LAYERS` selects six of the seven weight matrices in every block: the four attention projections and the two that read the MLP's input. The seventh (`down_proj`) reads a much wider input and would triple the time on this GPU; set `LAYERS = None` to include it if you have longer.

In [ ]:
import itertools, os, textwrap, time

import matplotlib.pyplot as plt
import torch

import rspd
from rspd import SRSPD

MODEL = "Qwen/Qwen3-4B-Instruct-2507"
TARGET_TOKENS = 1_000          # Dutch, to remove
RETAINED_TOKENS = 20_000       # English, German and chat, to keep
HELD_OUT_TOKENS = 2_000        # per language, for measuring; never shown to the solve
DOSES = [0.003, 0.01]          # every dose is solved from one pass over the data
DOSE = 0.003                   # the one applied first
LAYERS = {"types": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj"]}

# The test suite sets this to run every cell on a tiny random model with made-up text.
SMOKE = os.environ.get("RSPD_EXAMPLES_SMOKE") == "1"

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
COLORS = {"nl": "#c43d6f", "en": "#3f6fd1", "de": "#b98a00", "fr": "#4f9d69"}
NAMES = {"nl": "Dutch", "en": "English", "de": "German", "fr": "French"}

## Load the model

In [ ]:
if SMOKE:
    import rspd.toy
    model, tokenizer = rspd.toy.tiny_language_model()
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    assert torch.cuda.is_available(), "No GPU: on Colab, Runtime -> Change runtime type -> T4 GPU"
    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map="cuda").eval()

print(f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters on {model.device}")

## The two sets of text

Dutch Wikipedia is the target. The retained set is English and German Wikipedia plus chat conversations, so that the model's other languages and its chat behaviour count as things to keep. German is in there on purpose: it is the language closest to Dutch, and the one most at risk.

A separate slice of each language is held out for measuring.

In [ ]:
def wikipedia(lang, skip=0):
    from datasets import load_dataset
    rows = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    return (r["text"] for r in itertools.islice(rows, skip, None) if len(r["text"]) > 2000)


def chats():
    from datasets import load_dataset
    rows = load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft", streaming=True)
    return (tokenizer.apply_chat_template(r["messages"], tokenize=False) for r in rows)


def take(texts, n_tokens, length=512):
    """Token sequences of at most `length` tokens, `n_tokens` in total."""
    out, n = [], 0
    for text in texts:
        ids = tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids[0]
        for i in range(0, len(ids), length):
            piece = ids[i:i + length][: n_tokens - n]
            if len(piece) >= 16:
                out.append(piece)
                n += len(piece)
            if n >= n_tokens:
                return out
    return out


if SMOKE:
    toy = rspd.toy.tiny_texts(n=40)
    TARGET_TOKENS, RETAINED_TOKENS, HELD_OUT_TOKENS = 200, 1200, 200
    source = {lang: iter(toy[lang][:30]) for lang in toy} | {"chat": iter(toy["en"][:30])}
    held_source = {lang: iter(toy[lang][30:]) for lang in toy}
else:
    source = {"nl": wikipedia("nl"), "en": wikipedia("en"), "de": wikipedia("de"), "chat": chats()}
    held_source = {lang: wikipedia(lang, skip=3000) for lang in ("nl", "en", "de", "fr")}

target = take(source["nl"], TARGET_TOKENS)
retained = (take(source["en"], int(0.4 * RETAINED_TOKENS))
            + take(source["de"], int(0.3 * RETAINED_TOKENS))
            + take(source["chat"], int(0.3 * RETAINED_TOKENS)))
held_out = {lang: take(held_source[lang], HELD_OUT_TOKENS) for lang in ("nl", "en", "de", "fr")}

print(f"target:   {sum(map(len, target)):>6,} tokens of Dutch")
print(f"retained: {sum(map(len, retained)):>6,} tokens of English, German and chat")

## Before

Two measurements, used throughout:

- **Loss** on held-out text in each language: how surprised the model is by the next token. Higher means it reads that language worse.
- **What it says** when asked something in each language.

In [ ]:
PROMPTS = {
    "nl": "Schrijf drie zinnen over de stad Amsterdam.",
    "en": "Write three sentences about the city of London.",
    "de": "Schreibe drei Sätze über die Stadt Berlin.",
}
if SMOKE:
    PROMPTS = {lang: toy[lang][0] for lang in ("nl", "en", "de")}


@torch.no_grad()
def loss(sequences):
    total = count = 0
    for ids in sequences:
        ids = ids.unsqueeze(0).to(model.device)
        logits = model(ids).logits[0, :-1].float()
        total += float(torch.nn.functional.cross_entropy(logits, ids[0, 1:], reduction="sum"))
        count += ids.shape[1] - 1
    return total / count


@torch.no_grad()
def say(prompt, max_new_tokens=60):
    inputs = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], return_tensors="pt",
                                           add_generation_prompt=True, return_dict=True).to(model.device)
    out = model.generate(**inputs, max_new_tokens=8 if SMOKE else max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def measure():
    return {"loss": {lang: loss(seqs) for lang, seqs in held_out.items()},
            "says": {lang: say(p) for lang, p in PROMPTS.items()}}


def show(result):
    print("loss:  " + "   ".join(f"{NAMES[k]} {v:.2f}" for k, v in result["loss"].items()), end="\n\n")
    for lang, text in result["says"].items():
        print(f"[{NAMES[lang]}] {PROMPTS[lang]}")
        print(textwrap.indent(textwrap.fill(" ".join(text.split()), 96), "    "), end="\n\n")


before = measure()
show(before)

## Fit

`SRSPD(model, tokenizer)` attaches to the model. `fit` runs the two sets through it, collects the activations entering every selected matrix, and solves for each matrix's component. The model is not changed here.

This is the slow cell: about ten minutes on a T4. The solve works in double precision, which this class of GPU does slowly.

In [ ]:
def progress(event, info):
    if event in ("plan", "pass_done"):
        print(f"  {time.time() - start:5.0f}s  {event}  " +
              ", ".join(f"{k}={v}" for k, v in info.items() if isinstance(v, (int, str))))


start = time.time()
srspd = SRSPD(model, tokenizer, layers=LAYERS, scratch_dir="rspd-scratch", progress=progress)
components = srspd.fit(target=target, retained=retained, mu_per_token=DOSES, ridge_rel=1e-10,
                       eig_extremes=False)   # skips one diagnostic to save a few minutes here

print(f"\n{len(components)} components in {(time.time() - start) / 60:.1f} minutes")
if torch.cuda.is_available():
    print(f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")

Each component is a pair of factors for one weight matrix, `M = A @ B`, with its **leaks**: the share of the layer's response to Dutch that survives subtracting it (`L_X_rel`, lower removes more), and the share of its response to the retained set that is disturbed (`L_Y_rel`, lower keeps more).

In [ ]:
components = components.at(DOSE)
c = components[components.names[len(components) // 2]]
print(c.name, " A", tuple(c.A.shape), " B", tuple(c.B.shape), " leaks", {k: round(v, 3) for k, v in c.leaks.items()})

leaks = components.leaks
kinds = sorted({name.rsplit(".", 1)[-1] for name in leaks})
median = lambda xs: sorted(xs)[len(xs) // 2]
print(f"\n{'matrix type':<12}{'Dutch surviving':>18}{'retained disturbed':>21}")
for kind in kinds:
    rows = [v for name, v in leaks.items() if name.endswith(kind)]
    print(f"{kind:<12}{median([r['L_X_rel'] for r in rows]):>18.3f}{median([r['L_Y_rel'] for r in rows]):>21.3f}")

## Subtract, and measure again

`apply()` writes `W - M` into every selected matrix.

In [ ]:
components.apply(DOSE)
after = measure()
show(after)

In [ ]:
langs = list(before["loss"])
x = range(len(langs))
fig, ax = plt.subplots(figsize=(6.4, 3.0))
ax.bar([i - 0.2 for i in x], [before["loss"][k] for k in langs], width=0.4, color="#b5bac4", label="before")
ax.bar([i + 0.2 for i in x], [after["loss"][k] for k in langs], width=0.4,
       color=[COLORS[k] for k in langs], label="after")
ax.set_xticks(list(x), [NAMES[k] for k in langs])
ax.set_ylabel("loss on held-out text")
ax.set_title("Dutch is gone; the other languages are where they were", loc="left")
ax.legend(frameon=False, ncol=2, loc="lower right", bbox_to_anchor=(1.0, 1.0))
plt.show()

## The dose

Both doses were solved in the one `fit`. Applying another takes seconds, because nothing is recomputed: the previous edit is undone and the new one written.

In [ ]:
by_dose = {}
for dose in DOSES:
    components.apply(dose)
    by_dose[dose] = {lang: loss(seqs) for lang, seqs in held_out.items()}
    print(f"mu_per_token {dose:<6}  " + "   ".join(f"{NAMES[k]} {v:.2f}" for k, v in by_dose[dose].items()))

fig, ax = plt.subplots(figsize=(6.4, 3.0))
for lang in langs:
    ys = [before["loss"][lang]] + [by_dose[d][lang] for d in DOSES]
    ax.plot(range(len(ys)), ys, color=COLORS[lang], lw=2, marker="o", ms=4)
    ax.annotate(NAMES[lang], (len(ys) - 1, ys[-1]), xytext=(8, 0), textcoords="offset points",
                va="center", color=COLORS[lang], fontsize=9)
ax.set_xticks(range(len(DOSES) + 1), ["none"] + [str(d) for d in DOSES])
ax.set_xlabel("mu_per_token")
ax.set_ylabel("loss on held-out text")
ax.set_title("A larger dose removes more Dutch, and costs the others more", loc="left")
plt.show()

## Put it back

`restore()` returns every weight to its original value, bit for bit.

In [ ]:
components.restore()
restored = {lang: loss(seqs) for lang, seqs in held_out.items()}
drift = max(abs(restored[k] - before["loss"][k]) for k in restored)
assert drift < 1e-4, (restored, before["loss"])
print(f"restored: every loss is back where it started (largest difference {drift:.1e})")

## Keep the edited model

To save a model with Dutch removed, apply the dose you want and export it. The result loads with `from_pretrained` like any other checkpoint.

```python
components.apply(0.003)
rspd.export_checkpoint(model, tokenizer, "qwen3-4b-no-dutch")
```

## Things to try

- **Another language or topic.** Change the target: `wikipedia("fr")`, or your own list of strings.
- **A different trade.** More retained text makes the edit more careful with what it keeps; the solve time grows with it.
- **Every matrix.** `LAYERS = None` includes `down_proj`. It fits in 16 GB and takes about 30 minutes on a T4.
- **Smaller components.** `srspd.fit(..., rank_rule="erank")` also solves a low-rank version of every component.

In [ ]:
components.close()